# 04 · Validate and explore
Row counts, null rates, Iceberg metadata tables, time travel, and a first analysis.

> **Schema assumption:** column names follow the classic BTS / ASA Data Expo airline layout
> (`Year, Month, DayOfWeek, CRSDepTime, UniqueCarrier, Origin, Dest, Distance, ArrDelay, DepDelay,
> Cancelled, Diverted, ...`; carriers keyed by `Code`/`Description`). If your CSV headers differ,
> edit the constants in the first code cell.

In [ ]:
from pyspark.sql import SparkSession

# Catalog config (REST catalog + MinIO) comes from spark-defaults.conf.
spark = SparkSession.builder.appName("04-validate").getOrCreate()
spark.sparkContext.setLogLevel("WARN")
print("Spark", spark.version, "| UI:", spark.sparkContext.uiWebUrl)

NS = "local.airline"
DELAY_COLS = ["ArrDelay", "DepDelay", "CarrierDelay", "WeatherDelay", "NASDelay",
              "SecurityDelay", "LateAircraftDelay"]
CARRIER_KEY, CARRIER_NAME = "Code", "Description"     # in carriers
FLIGHT_CARRIER = "UniqueCarrier"                       # in flights

## Row counts

In [ ]:
for t in ["flights", "airports", "carriers", "plane_types"]:
    print(f"{NS}.{t:<12} {spark.table(f'{NS}.{t}').count():>12,}")

## Null rates on delay columns
BTS leaves delay-cause columns empty for on-time flights, so high null rates there are expected.

In [ ]:
from pyspark.sql import functions as F

flights = spark.table(f"{NS}.flights")
cols = [c for c in DELAY_COLS if c in flights.columns]
total = flights.count()
nulls = flights.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in cols]).first().asDict()
for c in cols:
    print(f"{c:<20} {nulls[c] / total:6.1%} null")

## Iceberg metadata tables

In [ ]:
spark.sql(f"SELECT partition, record_count, file_count FROM {NS}.flights.partitions ORDER BY partition").show()
spark.sql(f"SELECT snapshot_id, operation, committed_at FROM {NS}.flights.snapshots").show(truncate=False)

## Time travel

In [ ]:
snap = spark.sql(f"SELECT snapshot_id FROM {NS}.flights.snapshots ORDER BY committed_at").first()[0]
n = spark.sql(f"SELECT COUNT(*) AS n FROM {NS}.flights VERSION AS OF {snap}").first()["n"]
print(f"rows at first snapshot {snap}: {n:,}")

## Analysis: average arrival delay by carrier

In [ ]:
top = (flights.filter("Cancelled = 0")
       .groupBy(FLIGHT_CARRIER).agg(F.count("*").alias("flights"), F.round(F.avg("ArrDelay"), 2).alias("avg_arr_delay"))
       .filter("flights > 1000"))
carriers = spark.table(f"{NS}.carriers").select(F.col(CARRIER_KEY).alias(FLIGHT_CARRIER), F.col(CARRIER_NAME).alias("carrier"))
top.join(carriers, FLIGHT_CARRIER, "left").orderBy(F.desc("avg_arr_delay")).show(10, truncate=False)

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

by_month = (flights.filter("Cancelled = 0").groupBy("Month")
            .agg(F.avg("ArrDelay").alias("avg_arr_delay")).orderBy("Month").toPandas())
ax = by_month.plot(x="Month", y="avg_arr_delay", kind="bar", legend=False, figsize=(8, 3))
ax.set_ylabel("avg arrival delay (min)")
plt.tight_layout()
plt.show()